# English Accent Detection

Detects English accents from speech audio using deep learning.
Given any audio or video file, the system identifies whether the
speaker has an American, British, Australian, or Indian accent.

## My approach
- **Dataset:** Speech Accent Archive — 2,140 speakers from 177 countries,
  all reading the same standardized paragraph
- **Feature extraction:** Facebook Wav2Vec2 (94M parameters, pretrained
  on 960 hours of English speech)
- **Classifier:** Logistic Regression with StandardScaler pipeline
- **Final accuracy:** 75% on held-out test data

## tools
- Wav2Vec2 works excellently with small datasets through transfer learning —
  I only needed 210 labeled samples instead of thousands
- Logistic regression prevents overfitting on small datasets better than
  deep neural networks
- Speech Accent Archive has controlled recordings — every speaker reads
  the exact same paragraph, so the only variable is accent

## Project flow
```
Audio file → ffmpeg extracts WAV → Wav2Vec2 creates 768-number fingerprint
→ Logistic Regression predicts accent → result with confidence score
```


In [ ]:
import sys
import platform

print("Python version:", sys.version)
print("Operating system:", platform.system())

import subprocess
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print("\nGPU: YES — you have a GPU, model will run fast")
else:
    print("\nGPU: None — using CPU, model will be slower but still works")

In [ ]:
!pip install transformers torchaudio datasets -q
print("Done")

In [ ]:
import torch
import torchaudio
import numpy as np
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("torchversion: ", torch.__version__)
print("torchaudion:", torchaudio.__version__)
print("CUDa available:", torch.cuda.is_available())
print("Device:","GPU" if torch.cuda.is_available() else "CPU")

print(" All imports sucessful!")


In [ ]:
print("Downloading Wav2Vec2 from Hugging Face...")
print("First time: ~2 minutes (360MB download)")
print("After that: instant (loads from cache)\n")

MODEL_NAME='facebook/wav2vec2-base-960h'

processor= Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model= Wav2Vec2Model.from_pretrained(MODEL_NAME)

device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
model= model.to(device)
model.eval()

total_params= sum(p.numel() for p in model.parameters())
print(f"model loaded on:{device}")
print(f"Total parameters: {total_params:,}")
print(f"That means: {total_params/1_000_000:.1f} million numbers this model learned")

In [ ]:
import matplotlib.pyplot as plt

# NOT using real speech yet
SAMPLE_RATE= 16000
DURATION= 3

# Generate time axis
t= torch.linspace(0, DURATION, SAMPLE_RATE * DURATION)

# Mix frequencies to simulate speech-like sound
# Real speech is many frequencies mixed together

audio= (
    0.5 * torch.sin(2 * 3.14159 *200 * t) + # low frequency (voice base)
    0.3 * torch.sin(2 * 3.14159 * 800 *t) + # mid frequency (vowels)
    0.15 * torch.sin(2 * 3.14159 * 2000 *t) + # high frequency (consonants)
    0.05 * torch.randn(len(t))               # noise (background)

)
# Normalize between -1 and 1
audio= audio/ audio.abs().max()
print("=== What audio IS as data ===\n")
print(f"Duration: {DURATION} seconds")
print(f"Sample rate: {SAMPLE_RATE} samples/second")
print(f"Total numbers in this audio: {len(audio):,}")
print(f"\nFirst 10 numbers: {[round(x.item(),4) for x in audio[:10]]}")
print(f"Min value: {audio.min():.4f}")
print(f"Max value: {audio.max():.4f}")
print(f"\nThink of it as: {len(audio):,} numbers, each describing")
print(f"how much the air moved at that exact millisecond")

# Plot it so you can SEE the audio
plt.figure(figsize=(12, 3))
plt.plot(t[:1600].numpy(), audio[:1600].numpy(), color='steelblue', linewidth=0.8)
plt.title("What audio looks like — first 0.1 seconds (1600 samples)")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (air movement)")
plt.tight_layout()
plt.show()

print("\n✅ This wave shape is what Wav2Vec2 reads as input")


In [ ]:
print("=== Passing audio through Wav2Vec2 ===\n")

# 1:Format the audio using the processor
# The processor does two things:
# - Normalizes the audio (makes volume consistent)
# - Converts it to a tensor format the model expects
inputs = processor(
    audio.numpy(),           # convert PyTorch tensor to numpy
    sampling_rate=16000,     # audio is 16kHz
    return_tensors="pt",     # return as PyTorch tensors
    padding=True
)

print("After processor:")
print(f"  Input shape: {inputs.input_values.shape}")
print(f"  This means: {inputs.input_values.shape[0]} audio clip, {inputs.input_values.shape[1]} samples\n")

# 2:Move to GPU and run through the neural network
input_values = inputs.input_values.to(device)

with torch.no_grad():       # no_grad = don't calculate gradients (saves memory)
    outputs = model(input_values)

print("After Wav2Vec2 neural network:")
print(f"  Output shape: {outputs.last_hidden_state.shape}")
print(f"  This means: {outputs.last_hidden_state.shape[0]} clip, "
      f"{outputs.last_hidden_state.shape[1]} time steps, "
      f"{outputs.last_hidden_state.shape[2]} features\n")

# 3: Average across time steps to get ONE fixed vector
# Averaging gives us one consistent size regardless of clip length
embedding = outputs.last_hidden_state.mean(dim=1)

print("After averaging across time:")
print(f"  Embedding shape: {embedding.shape}")
print(f"  This is the FINGERPRINT: {embedding.shape[1]} numbers describing this speech\n")

# Show the actual numbers
emb_numpy = embedding.cpu().numpy()[0]
print(f"First 10 numbers of the fingerprint:")
print(f"  {[round(x, 4) for x in emb_numpy[:10]]}")
print(f"\nMin value: {emb_numpy.min():.4f}")
print(f"Max value: {emb_numpy.max():.4f}")
print(f"Mean value: {emb_numpy.mean():.4f}")
print("\n This 768-number vector is what your classifier will learn from")

In [ ]:
def get_embedding(audio_tensor):

     # Format for Wav2Vec2
    inputs = processor(
        audio_tensor.numpy(),
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

     # Move to GPU
    input_values = inputs.input_values.to(device)

    # Run through neural network — no gradient tracking needed
    with torch.no_grad():
        outputs = model(input_values)
     # Average across time steps → shape [1, 768]
    embedding = outputs.last_hidden_state.mean(dim=1)

    # Return as flat numpy array of 768 numbers
    # [0] removes the batch dimension: [1, 768] → [768]
    return embedding.cpu().numpy()[0]

# ── Test the function works
print("Testing get_embedding function...\n")

test_result = get_embedding(audio)

print(f"Input:  audio tensor of {len(audio):,} samples")
print(f"Output: numpy array of {len(test_result)} numbers")
print(f"Type:   {type(test_result)}")
print(f"Shape:  {test_result.shape}")
print(f"\n Function works correctly")
print(f"\nThis function will be called once per training sample")
print(f"and once every time a user uploads a video to your app")

In [ ]:
import os
import json
from google.colab import drive


drive.mount('/content/drive')

KAGGLE_USERNAME = "your_kaggle_username"
KAGGLE_KEY      = "your_kaggle_key"


os.makedirs("/root/.config/kaggle", exist_ok=True)
with open("/root/.config/kaggle/kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)

# Download dataset
print("Redownloading Speech Accent Archive...")
!kaggle datasets download -d rtatman/speech-accent-archive \
    -p /content/accent_data --unzip -q

print(" Dataset ready!")
print(os.listdir("/content/accent_data"))

In [ ]:
import pandas as pd

#  Look inside the recordings folder
recordings_path = "/content/accent_data/recordings"
audio_files = os.listdir(recordings_path)

print("=== Dataset Overview ===\n")
print(f"Total audio files: {len(audio_files)}")
print(f"First 10 files: {audio_files[:10]}")

#  Load the CSV metadata file
df = pd.read_csv("/content/accent_data/speakers_all.csv")

print(f"\n=== Metadata CSV ===")
print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
print(df.head())

#  See what accents/languages are in this dataset
print(f"\n=== Native Languages (top 20) ===")
print(df['native_language'].value_counts().head(20))

In [ ]:
import os

# Find where the actual audio files are
print("=== Searching for audio files ===\n")

# Walk through ALL folders and subfolders
for root, dirs, files in os.walk("/content/accent_data"):
    mp3_files = [f for f in files if f.endswith('.mp3')]
    if mp3_files:
        print(f"Found {len(mp3_files)} MP3 files in: {root}")
        print(f"First 5: {mp3_files[:5]}\n")
        AUDIO_PATH = root
        break

print(f"\n Audio files are at: {AUDIO_PATH}")

In [ ]:
import pandas as pd

df = pd.read_csv("/content/accent_data/speakers_all.csv")

# Clean the dataframe first
# Drop the empty unnamed columns
df = df[['age', 'age_onset', 'birthplace', 'filename',
         'native_language', 'sex', 'speakerid', 'country', 'file_missing?']]

# Fill missing values
df['native_language'] = df['native_language'].str.strip().str.lower()
df['birthplace'] = df['birthplace'].str.strip().str.lower()
df['country'] = df['country'].str.strip().str.lower()

print("=== Building Accent Labels ===\n")
print("Strategy: use native_language + country to assign accent\n")

# Define which rows belong to which accent

def assign_accent(row):
    lang = str(row['native_language']).lower()
    country = str(row['country']).lower()
    birthplace = str(row['birthplace']).lower()

    # American
    if lang == 'english' and country in ['usa', 'united states']:
        return 'american'

    # British
    if lang == 'english' and country in ['uk', 'england', 'scotland',
                                          'wales', 'northern ireland']:
        return 'british'

    # Australian
    if lang == 'english' and country in ['australia']:
        return 'australian'

    # Indian
    if country == 'india':
        return 'indian'

    # Irish
    if lang == 'english' and country in ['ireland']:
        return 'irish'

    return None
df['accent'] = df.apply(assign_accent, axis=1)

#  check how many samples we got per accent
accent_counts = df['accent'].value_counts()
print("Samples per accent BEFORE removing missing files:")
print(accent_counts)

# Remove rows where audio file is marked missing
df_clean = df[df['file_missing?'] != True].copy()
df_accent = df_clean[df_clean['accent'].notna()].copy()

print(f"\nSamples per accent AFTER cleaning:")
print(df_accent['accent'].value_counts())

print(f"\nTotal usable samples: {len(df_accent)}")
print(f"\nExample rows:")
print(df_accent[['filename', 'native_language', 'country',
                  'birthplace', 'accent']].head(10))

In [ ]:
print("=== Fixing Class Imbalance ===\n")
# Check current distribution
print("Before balancing:")
print(df_accent['accent'].value_counts())

# Decide on sample size per class
SAMPLES_PER_CLASS = 60

df_balanced = df_accent.groupby('accent').apply(
    lambda x: x.sample(
        min(len(x), SAMPLES_PER_CLASS),
        random_state=42
    )
).reset_index(drop=True)

print(f"\nAfter balancing (max {SAMPLES_PER_CLASS} per class):")
print(df_balanced['accent'].value_counts())
print(f"\nTotal samples: {len(df_balanced)}")

#  Verify the audio files actually exist
print("\nVerifying audio files exist on disk...")

def get_audio_path(filename):
    return f"{AUDIO_PATH}/{filename}.mp3"

df_balanced['audio_path'] = df_balanced['filename'].apply(get_audio_path)
df_balanced['file_exists'] = df_balanced['audio_path'].apply(os.path.exists)

missing = df_balanced[df_balanced['file_exists'] == False]
print(f"Files found: {df_balanced['file_exists'].sum()}")
print(f"Files missing: {len(missing)}")

if len(missing) > 0:
    print(f"Missing filenames: {missing['filename'].tolist()}")

# Keep only rows where file exists
df_final = df_balanced[df_balanced['file_exists'] == True].copy()
print(f"\n Final dataset ready: {len(df_final)} samples")
print(f"\nFinal distribution:")
print(df_final['accent'].value_counts())

In [ ]:
import torchaudio
import numpy as np
from tqdm import tqdm

print("=== Extracting Real Speech Embeddings ===")
print("Wav2Vec2 will listen to each audio file and create a 768-number fingerprint")
print(f"Processing {len(df_final)} files... (~5-10 minutes)\n")

embeddings = []   # store the 768-number fingerprint for each file
labels = []       # store the accent label for each file
failed = []       # store any files that fail to process

for idx, row in tqdm(df_final.iterrows(), total=len(df_final)):
    try:
        # Step 1: Load the MP3 file
        # torchaudio.load reads the audio file from disk
        # speech_array shape: (channels, samples)
        # sample_rate: how many samples per second in THIS file
        speech_array, sample_rate = torchaudio.load(row['audio_path'])

        #  Step 2: Convert stereo to mono if needed
        # Some recordings are stereo (2 channels)
        # Wav2Vec2 needs mono (1 channel)

        if speech_array.shape[0] > 1:
            speech_array = speech_array.mean(dim=0, keepdim=True)

        # Step 3: Resample to 16kHz if needed
        # This dataset has files at different sample rates
        # Wav2Vec2 was trained on 16kHz — must match
        if sample_rate != 16000:
            resampler = torchaudio.transforms.Resample(
                orig_freq=sample_rate,
                new_freq=16000
            )
            speech_array = resampler(speech_array)

        # Step 4: Remove channel dimension
        # squeeze: (1, samples) → (samples,)
        # get_embedding expects a flat 1D tensor
        speech_tensor = speech_array.squeeze()

        # Step 5: Get the 768-number fingerprint
        embedding = get_embedding(speech_tensor)

        embeddings.append(embedding)
        labels.append(row['accent'])

    except Exception as e:
        # If a file fails for any reason, skip it and record why
        failed.append({'file': row['filename'], 'error': str(e)})
        continue

# Convert to numpy arrays
X = np.array(embeddings)   # shape: (221, 768)
y = np.array(labels)       # shape: (221,)

print(f"\n Embedding extraction complete!")
print(f"\nSuccessfully processed: {len(embeddings)} files")
print(f"Failed: {len(failed)} files")
if failed:
    print(f"Failed files: {failed}")

print(f"\nX shape: {X.shape}")
print(f"  → {X.shape[0]} audio samples")
print(f"  → {X.shape[1]} features per sample (the Wav2Vec2 fingerprint)")
print(f"\ny shape: {y.shape}")
print(f"  → {y.shape[0]} accent labels")
print(f"\nLabel distribution:")
unique, counts = np.unique(y, return_counts=True)
for accent, count in zip(unique, counts):
    print(f"  {accent}: {count} samples")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("=== Training the Accent Classifier ===\n")

# Step 1: Split into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {len(y_train)} samples  ← model learns from these")
print(f"Testing set:  {len(y_test)} samples   ← model is evaluated on these")
print(f"\nTraining distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for accent, count in zip(unique, counts):
    print(f"  {accent}: {count}")

# Step 2: Build the pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        solver='lbfgs',
        multi_class='multinomial'
    ))
])

# Step 3: Train
print(f"\nTraining logistic regression...")
pipeline.fit(X_train, y_train)
print(" Training complete!")

# Step 4: Evaluate
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n=== Results on Test Set ===")
print(f"Overall accuracy: {accuracy*100:.1f}%")
print(f"\nDetailed breakdown:")
print(classification_report(y_test, y_pred))

# Step 5: Confusion matrix
accent_names = sorted(list(set(y)))
cm = confusion_matrix(y_test, y_pred, labels=accent_names)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=accent_names,
    yticklabels=accent_names
)
plt.title('Confusion Matrix — Where does the model get confused?')
plt.ylabel('Actual Accent')
plt.xlabel('Predicted Accent')
plt.tight_layout()
plt.show()

In [ ]:
print("=== Understanding Why the Model Struggles ===\n")

print("Current sample counts:")
unique, counts = np.unique(y, return_counts=True)
for accent, count in zip(unique, counts):
    status = "enough" if count >= 50 else "⚠️ borderline" if count >= 25 else "❌ too few"
    print(f"  {accent:<12} {count:>3} samples  {status}")

print("""
Rule of thumb in machine learning:
  < 20 samples  → model cannot learn this class reliably
  20-50 samples → model learns weak patterns
  50-200 samples → model learns decent patterns
  200+ samples  → model learns strong patterns

Your irish class has 11 samples. That explains f1=0.00.
Your australian class has 32 samples. That explains f1=0.43.
""")

print("=== What Would Actually Fix This ===\n")
print("Option 1: Get more Irish and Australian samples")
print("  → Mozilla Common Voice has thousands of labeled accent samples")
print("  → accentdb dataset has more Indian English samples")
print()
print("Option 2: Remove Irish from the model (not enough data)")
print("  → Build a 4-accent model: American, British, Indian, Australian")
print("  → More honest than having a class that always predicts wrong")
print()
print("Option 3: Fine-tune Wav2Vec2 itself on accent data")
print("  → Instead of frozen features + logistic regression")
print("  → Update all 94M parameters on accent-labeled data")
print("  → Requires GPU and ~1000+ samples per class")
print("  → Expected accuracy: 85-92%")
print()

#  Let's try Option 2 right now — 4 accent model
print("=== Training 4-Accent Model (removing Irish) ===\n")

# Filter out Irish
mask = y != 'irish'
X_4 = X[mask]
y_4 = y[mask]

print(f"Samples after removing Irish: {len(y_4)}")
print(f"Distribution: { {a:int(c) for a,c in zip(*np.unique(y_4, return_counts=True))} }")

X_train4, X_test4, y_train4, y_test4 = train_test_split(
    X_4, y_4,
    test_size=0.2,
    random_state=42,
    stratify=y_4
)

pipeline4 = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        solver='lbfgs',
        multi_class='multinomial'
    ))
])

pipeline4.fit(X_train4, y_train4)
y_pred4 = pipeline4.predict(X_test4)
accuracy4 = accuracy_score(y_test4, y_pred4)

print(f"\n4-accent model accuracy: {accuracy4*100:.1f}%")
print(f"5-accent model accuracy: 66.7%")
print(f"Improvement: +{(accuracy4-0.667)*100:.1f}%")
print(f"\nDetailed breakdown:")
print(classification_report(y_test4, y_pred4))

## Improving the Model — Fixing British Accent Confusion

After training the first model I discovered British accent was being
confused with American (f1=0.50). I investigated and found two problems:

**Problem 1 — Overfitting**
The model predicted British perfectly on training data (48/48 correct, 98.6%
confidence) but failed on new British audio. Classic overfitting — it memorized
the 48 training speakers instead of learning general British patterns.

**Problem 2 — Regional diversity**
The 65 British samples came from ALL across the UK:
- London, Oxford, Manchester → standard English accent
- Glasgow, Edinburgh → Scottish accent (very different)
- Cardiff → Welsh accent (very different)
- Belfast → Northern Irish accent (very different)

All labeled "british" but sounding completely different. The model learned
4 very different accent groups under one label and got confused.

**My fix:** Filter to England-only speakers. Remove Scottish, Welsh,
and Northern Irish samples. This creates a tighter, more consistent
British cluster in embedding space.

**Result:** Accuracy improved from 69% to 75% — without collecting any new data.

**Key lesson:** Better data quality beats better algorithms every time.

In [ ]:
import pandas as pd
print("=== Solution 1: Filter British to Southern England only ===\n")
print("Problem: British samples span all UK regions — too diverse")
print("Fix: Keep only speakers from England (southern RP accent)")
print("This makes British cluster tighter in embedding space\n")

# Reload full dataframe
df = pd.read_csv("/content/accent_data/speakers_all.csv")
df['native_language'] = df['native_language'].str.strip().str.lower()
df['birthplace'] = df['birthplace'].str.strip().str.lower()
df['country'] = df['country'].str.strip().str.lower()

# Reassign accents same as before
def assign_accent(row):
    lang = str(row['native_language']).lower()
    country = str(row['country']).lower()
    if lang == 'english' and country in ['usa', 'united states']:
        return 'american'
    if lang == 'english' and country in ['uk', 'england']:
        # Only England this time — not scotland, wales, northern ireland
        return 'british'
    if lang == 'english' and country in ['australia']:
        return 'australian'
    if country == 'india':
        return 'indian'
    return None

df['accent'] = df.apply(assign_accent, axis=1)

# Check how many British samples we have now
print("British samples by birthplace (England only):")
british_df = df[df['accent'] == 'british']
print(british_df['birthplace'].value_counts().head(20))
print(f"\nTotal British samples: {len(british_df)}")
print(f"Previous British samples: 65 (all UK)")

In [ ]:
print("=== Filtering British by birthplace text ===\n")
print("Strategy: exclude birthplaces containing scotland, wales,")
print("northern ireland, glasgow, edinburgh, cardiff, belfast\n")

# Words that indicate non-English UK regions
exclude_words = [
    'scotland', 'scottish', 'glasgow', 'edinburgh', 'aberdeen',
    'wales', 'welsh', 'cardiff', 'swansea',
    'northern ireland', 'belfast', 'strabane',
    'ireland'
]

def is_english_british(row):
    lang = str(row['native_language']).lower()
    country = str(row['country']).lower()
    birthplace = str(row['birthplace']).lower()

    # Must be native English speaker from UK
    if lang != 'english' or country not in ['uk', 'england']:
        return False

    # Exclude Scottish, Welsh, Northern Irish birthplaces
    for word in exclude_words:
        if word in birthplace:
            return False

    return True

# Apply filter
british_england = df[df.apply(is_english_british, axis=1)]

print(f"All UK English speakers:        65")
print(f"England only (after filtering): {len(british_england)}")
print(f"\nBirthplaces kept:")
print(british_england['birthplace'].value_counts())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score
import pickle

print("=== Retraining with cleaner British samples ===\n")

# Build clean dataset
# need to re-extract embeddings for the filtered British samples
# and combine with our existing American, Australian, Indian embeddings

# get the filtered British speaker filenames
british_filenames = set(british_england['filename'].tolist())
print(f"Clean British speakers: {len(british_filenames)}")

# Get existing non-British embeddings and labels from saved data
non_british_mask = y_4 != 'british'
X_non_british = X_4[non_british_mask]
y_non_british = y_4[non_british_mask]
print(f"Non-British samples kept: {len(y_non_british)}")
print(f"  {dict(zip(*np.unique(y_non_british, return_counts=True)))}")

#  Re-extract embeddings for clean British samples only
print(f"\nRe-extracting embeddings for {len(british_filenames)} clean British samples...")
print("(Only British samples — rest already saved)\n")

import torchaudio
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from tqdm import tqdm

# Load Wav2Vec2
MODEL_NAME = "facebook/wav2vec2-base-960h"
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
wav2vec_model = Wav2Vec2Model.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
wav2vec_model = wav2vec_model.to(device)
wav2vec_model.eval()

AUDIO_PATH = "/content/accent_data/recordings/recordings"

def get_embedding(audio_tensor):
    inputs = processor(
        audio_tensor.numpy(),
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )
    input_values = inputs.input_values.to(device)
    with torch.no_grad():
        outputs = wav2vec_model(input_values)
    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.cpu().numpy()[0]

# Extract embeddings for clean British samples
british_embeddings = []
british_labels = []
failed = []

british_rows = british_england.copy()
british_rows['audio_path'] = british_rows['filename'].apply(
    lambda f: f"{AUDIO_PATH}/{f}.mp3"
)
british_rows = british_rows[british_rows['audio_path'].apply(os.path.exists)]
print(f"British audio files found: {len(british_rows)}")

for idx, row in tqdm(british_rows.iterrows(), total=len(british_rows)):
    try:
        speech_array, sample_rate = torchaudio.load(row['audio_path'])
        if speech_array.shape[0] > 1:
            speech_array = speech_array.mean(dim=0, keepdim=True)
        if sample_rate != 16000:
            resampler = torchaudio.transforms.Resample(
                orig_freq=sample_rate, new_freq=16000
            )
            speech_array = resampler(speech_array)
        speech_tensor = speech_array.squeeze()
        embedding = get_embedding(speech_tensor)
        british_embeddings.append(embedding)
        british_labels.append('british')
    except Exception as e:
        failed.append(row['filename'])
        continue

print(f"\nSuccessfully processed: {len(british_embeddings)} British samples")
print(f"Failed: {len(failed)}")

# Combine clean British with existing other accents
X_british_new = np.array(british_embeddings)
y_british_new = np.array(british_labels)

X_combined = np.vstack([X_non_british, X_british_new])
y_combined = np.concatenate([y_non_british, y_british_new])

print(f"\nFinal combined dataset: {len(y_combined)} samples")
print(f"Distribution: { {l: int(c) for l, c in zip(*np.unique(y_combined, return_counts=True))} }")

In [ ]:
print("=== Comparing old vs new British model ===\n")

# Trained new model with clean British samples
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X_combined, y_combined,
    test_size=0.2,
    random_state=42,
    stratify=y_combined
)

new_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        C=0.1,
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        solver='lbfgs'
    ))
])

new_pipeline.fit(X_train_new, y_train_new)
y_pred_new = new_pipeline.predict(X_test_new)
accuracy_new = accuracy_score(y_test_new, y_pred_new)

print(f"Old model accuracy (all UK):       69.0%")
print(f"New model accuracy (England only): {accuracy_new*100:.1f}%")

print(f"\nDetailed breakdown:")
print(classification_report(y_test_new, y_pred_new))

#  Cross validation for reliable comparison
print("=== Cross validation comparison ===\n")

old_scores = cross_val_score(
    Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            C=0.1, max_iter=1000,
            class_weight='balanced', random_state=42
        ))
    ]),
    X_4, y_4, cv=5, scoring='accuracy'
)

new_scores = cross_val_score(
    Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            C=0.1, max_iter=1000,
            class_weight='balanced', random_state=42
        ))
    ]),
    X_combined, y_combined, cv=5, scoring='accuracy'
)

print(f"Old model (all UK British):")
print(f"  avg: {old_scores.mean()*100:.1f}%  "
      f"±{old_scores.std()*100:.1f}%")
print(f"  scores: {[round(s*100,1) for s in old_scores]}")

print(f"\nNew model (England only British):")
print(f"  avg: {new_scores.mean()*100:.1f}%  "
      f"±{new_scores.std()*100:.1f}%")
print(f"  scores: {[round(s*100,1) for s in new_scores]}")

print(f"\nImprovement: "
      f"+{(new_scores.mean()-old_scores.mean())*100:.1f}%")

In [ ]:
print("=== Saving improved model to Drive ===\n")

save_folder = '/content/drive/MyDrive/accent_detector'

# Save new model
model_path = f'{save_folder}/accent_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(new_pipeline, f)
print(f" New model saved: accent_model.pkl")
print(f"   Accuracy: 75.0% (was 69.0%)")

# Save new combined embeddings
embeddings_path = f'{save_folder}/embeddings.npz'
np.savez(
    embeddings_path,
    X=X_combined,
    y=y_combined
)
print(f" New embeddings saved: embeddings.npz")
print(f"   Samples: {len(y_combined)} (was 210)")

# Save labels
accent_labels = sorted(list(set(y_combined)))
labels_path = f'{save_folder}/accent_labels.pkl'
with open(labels_path, 'wb') as f:
    pickle.dump(accent_labels, f)
print(f" Labels saved: {accent_labels}")

# Save the CSV
import shutil
shutil.copy(
    "/content/accent_data/speakers_all.csv",
    f"{save_folder}/speakers_all.csv"
)
print(f" CSV saved to Drive permanently")

print(f"\n=== Summary of improvements made today ===")
print(f"  Started with:  69.0% accuracy, all UK British speakers")
print(f"  Discovered:    overfitting on regionally diverse British data")
print(f"  Fixed by:      filtering to England-only speakers")
print(f"  Result:        75.0% accuracy — +6% improvement")
print(f"  Key lesson:    better data beats better algorithms")

## Conclusions and Key Learnings

### Final results
| Accent | F1 Score | Samples | Notes |
|--------|----------|---------|-------|
| American | 0.89 | 60 | Best result — consistent accent, clean data |
| Indian | 0.80 | 58 | Very distinct phonetic features |
| Australian | 0.57 | 32 | Limited by small sample size |
| British | 0.57 | 48 | Improved after filtering to England-only |

**Overall accuracy: 75%** (improved from 69% baseline)

### Problems I discovered and fixed

**1. Class imbalance**
Original data had 373 American vs 11 Irish samples — 34:1 ratio.
Fixed by undersampling majority classes to 60 samples each and
using class_weight='balanced' in the classifier.

**2. Irish accent removed**
Only 11 Irish samples available — f1 score was 0.00.
More honest to remove the class than keep a broken predictor.

**3. British overfitting**
Model confused British with American on new audio.
Fixed by filtering to England-only speakers — removed Scottish,
Welsh and Northern Irish samples that were adding noise.
Accuracy improved +6% from data cleaning alone.

**4. Long audio clips**
Model failed on audio longer than 15 seconds.
Fixed by trimming to 15 seconds to match training data distribution.
Training samples were all 8-15 seconds — inference must match.

### What I would do with more time
1. Collect 500+ samples per accent from Mozilla Common Voice
2. Fine-tune Wav2Vec2 directly on accent-labeled data (expected 85-92% accuracy)
3. Add more accents — South African, New Zealand, Canadian
4. Build a confidence threshold — if model is less than 60% confident,
   say "accent unclear" instead of guessing

### What I learned
The most important lesson from this project was that **data quality
matters more than model complexity**. Switching from all-UK to
England-only British samples improved accuracy by 6% with zero
changes to the model. Understanding your data deeply is more
valuable than trying more complex algorithms.